# Halloween Orders: Data Quality and Exploratory Analysis

**Festa em Cena | Module 1, Week 07 | Mini-project**

This notebook investigates a fictional costume retailer's order records. We will
read one CSV, clean text, validate numbers and dates, test a business hypothesis,
and count registered orders by sales channel and year-month.

**One row is one order for one product, possibly containing several units.**
Order counts are not revenue, units sold, or completed sales.

## How to study and run this notebook

1. Follow the installation instructions in `README.md` and open this file.
2. Select the Python kernel from your `.venv` environment.
3. Start at the first cell and press **Shift+Enter** to run each cell in order.
4. Read the explanation, inspect the output, and explain the decision in your own words.
5. Before presenting, use **Restart Kernel and Run All Cells** to confirm reproducibility.

The five sprints below follow the assignment. Functions are defined before they
are called; intermediate tables let you inspect each transformation.
All explanations and working column names are English. The source CSV remains
unchanged. Required source labels (`sem categoria`, `Mercado`, `Loja física`,
`Web`) and observed values are retained to match the assignment exactly.


## Sprint 1 — Setup and ingestion

### 1. Import the tools

Pandas provides DataFrames. `re` handles regular expressions, `datetime` parses
dates, and `Decimal` validates numeric text without rounding it first.
`Path` locates the CSV in the project folder. `display` renders readable tables.


In [1]:
from pathlib import Path
from datetime import datetime
from decimal import Decimal, InvalidOperation
import re

import pandas as pd
from IPython.display import display

CSV_PATH = Path("halloween_sales.csv")
DATE_FORMAT = "%Y-%m-%d %H:%M:%S"
MISSING_CATEGORY = "sem categoria"

pd.set_option("display.max_columns", 20)


### 2. Load the English-header source

The input file `halloween_sales.csv` already has English column names. The
dictionary documents how the original Portuguese headers correspond to these
names; no translation is needed during loading. `load_orders` receives a file
path and returns a DataFrame. We explicitly use
commas, UTF-8, and string values. `keep_default_na=False` preserves empty strings
and literal text; our next function identifies actual blanks explicitly.

`raw_df` preserves the input values. `orders` is a separate copy for
transformations. The notebook reads only `halloween_sales.csv` and never
overwrites it. Its values include the deliberate errors for us to investigate.


In [2]:
COLUMN_NAMES = {
    "pedido_id": "order_id",
    "data_venda": "sale_date",
    "produto": "product",
    "categoria": "category",
    "canal_venda": "sales_channel",
    "cidade": "city",
    "quantidade": "quantity",
    "preco_unitario": "unit_price",
    "custo_unitario": "unit_cost",
    "desconto_pct": "discount_pct",
    "status_pedido": "order_status",
    "publico": "target_audience",
    "tamanho": "size",
    "data_entrega": "delivery_date",
}


def load_orders(path):
    """Read text values from the CSV, which already has English headers."""
    frame = pd.read_csv(
        path, sep=",", encoding="utf-8", dtype="string", keep_default_na=False
    )
    return frame


raw_df = load_orders(CSV_PATH)
orders = raw_df.copy(deep=True)

display(pd.DataFrame(COLUMN_NAMES.items(), columns=["source_column", "english_column"]))
print(f"Rows read: {len(raw_df):,}")
print(f"Distinct order IDs: {raw_df['order_id'].nunique():,}")
display(raw_df.head(3))


,source_column,english_column
0,pedido_id,order_id
1,data_venda,sale_date
2,produto,product
3,categoria,category
4,canal_venda,sales_channel
5,cidade,city
6,quantidade,quantity
7,preco_unitario,unit_price
8,custo_unitario,unit_cost
9,desconto_pct,discount_pct


Rows read: 1,500
Distinct order IDs: 1,500


,order_id,sale_date,product,category,sales_channel,city,quantity,unit_price,unit_cost,discount_pct,order_status,target_audience,size,delivery_date
0,844,2025-11-21 18:02:35,Fantasia de vampiro,Fantasia,Loja física,Curitiba,3,132.5,72.08,15,Cancelado,Adulto,G,
1,378,2024-10-29 18:41:26,Fantasia de bruxa,FANTASIA!!!,Loja física,Maringá,3,110,57,5,Concluído,Adulto,M,2024-11-03 18:41:26
2,953,2025-09-05 21:16:09,Fantasia de bruxa,Fantasia,Web,Maringá,2,116.6,60.42,5,Concluído,Adulto,M,2025-09-10 21:16:09


### 3. Understand the fields and review missing values

Four key fields are `order_id` (order identifier), `quantity` (units in an order),
`unit_price` (price per unit), and `sale_date` (local sale timestamp).
`order_status` describes the recorded status; `delivery_date` may be blank.
Costs and discounts are present but are optional and are not analyzed here.

`is_blank` handles both Pandas nulls and empty or whitespace-only text. Because
we preserved blanks on import, `.isna()` alone would miss empty strings.


In [3]:
def is_blank(value):
    """Return True for a null, an empty string, or whitespace-only text."""
    if pd.isna(value):
        return True
    return str(value).strip() == ""


missing_review = pd.DataFrame({
    "data_type": raw_df.dtypes.astype(str),
    "null_values": raw_df.isna().sum(),
    "blank_or_null_values": raw_df.apply(lambda column: column.map(is_blank).sum()),
})
display(missing_review)


,data_type,null_values,blank_or_null_values
order_id,string,0,0
sale_date,string,0,0
product,string,0,0
category,string,0,18
sales_channel,string,0,0
city,string,0,0
quantity,string,0,0
unit_price,string,0,12
unit_cost,string,0,7
discount_pct,string,0,0


## Sprint 2 — Text transformation

### 4. Clean categories

First trim surrounding spaces and lowercase the text. Then use `re.sub` to
remove punctuation, keeping letters and spaces. The source categories have no
accents, so `[a-z]` covers their letters. Collapse repeated whitespace while
retaining single spaces between words. If the result is empty, return the
required placeholder `sem categoria`.

Count originally blank categories separately from originally nonblank values
that changed. These two groups do not overlap. Comparisons use the original
text, so case and whitespace corrections count as changes too.


In [4]:
def clean_category(value):
    """Normalize a category without inventing a product classification."""
    if is_blank(value):
        return MISSING_CATEGORY
    text = str(value).strip().lower()
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    if text == "":
        return MISSING_CATEGORY
    return text


originally_blank_category = raw_df["category"].map(is_blank)
orders["category"] = raw_df["category"].map(clean_category)
changed_nonblank_category = (
    ~originally_blank_category & orders["category"].ne(raw_df["category"])
)

category_examples = pd.DataFrame({
    "before": raw_df["category"],
    "after": orders["category"],
}).drop_duplicates()

print(f"Originally blank categories filled: {originally_blank_category.sum()}")
print(f"Originally nonblank categories changed: {changed_nonblank_category.sum()}")
display(category_examples)


Originally blank categories filled: 18
Originally nonblank categories changed: 1482


,before,after
0,Fantasia,fantasia
1,FANTASIA!!!,fantasia
3,Acessorio,acessorio
12,Fantasia,fantasia
18,ACESSORIO!!!,acessorio
22,Acessorio,acessorio
39,,sem categoria


### 5. Standardize sales channels

A dictionary maps trimmed, lowercase text to the three required labels. An
unrecognized channel becomes a missing working value, while its original text
remains in `raw_df`. Such a row is excluded only from the channel count, not
from every analysis. No status filter is applied.


In [5]:
CHANNEL_NAMES = {
    "mercado": "Mercado",
    "loja física": "Loja física",
    "web": "Web",
}


def clean_channel(value):
    """Return a recognized label or a missing value for an unknown channel."""
    if is_blank(value):
        return pd.NA
    return CHANNEL_NAMES.get(str(value).strip().lower(), pd.NA)


orders["sales_channel"] = raw_df["sales_channel"].map(clean_channel).astype("string")
orders["unrecognized_channel"] = orders["sales_channel"].isna()
channel_changes = (
    orders["sales_channel"].notna()
    & orders["sales_channel"].ne(raw_df["sales_channel"])
).fillna(False)

print(f"Recognized channel values standardized: {channel_changes.sum()}")
print(f"Unrecognized channels: {orders['unrecognized_channel'].sum()}")
display(pd.DataFrame({
    "before": raw_df["sales_channel"],
    "after": orders["sales_channel"],
}).drop_duplicates())


Recognized channel values standardized: 45
Unrecognized channels: 0


,before,after
0,Loja física,Loja física
2,Web,Web
4,Mercado,Mercado
13,loja física,Loja física
104,mercado,Mercado
117,web,Web


## Sprint 3 — Missing values and numeric validation

### 6. Define explicit numeric rules

Quantity must be a finite, positive whole number. An exact numeric value such
as `2.0` represents an integer and is accepted; a fraction such as `2.5` is not.
Unit price must be a finite number greater than or equal to zero; zero is valid.
Empty, malformed, negative, or nonfinite values fail the relevant rule.

The functions return a number when valid and `None` otherwise. They do not
replace errors with zero, a mean, or another guessed value. `Decimal` is used
for validation; it is part of Python's standard library.


In [6]:
def parse_number(value):
    """Parse finite numeric text; return None when it cannot be used."""
    if is_blank(value):
        return None
    try:
        number = Decimal(str(value).strip())
    except InvalidOperation:
        return None
    if not number.is_finite():
        return None
    return number


def validate_quantity(value):
    """Accept only an integer quantity greater than zero."""
    number = parse_number(value)
    if number is None:
        return None
    if number <= 0 or number != number.to_integral_value():
        return None
    return int(number)


def validate_unit_price(value):
    """Accept a present, finite, nonnegative price, including zero."""
    number = parse_number(value)
    if number is None or number < 0:
        return None
    return number


### 7. Mark problems and inspect original examples

Validated numeric fields are added alongside the original text. An invalid
quantity or price remains visible and receives a Boolean flag. The union (`|`)
counts each affected order once, even if it has both problems. Adding the two
field counts could double-count an order.

No rows are deleted: channel and period counts can still use a row whose
quantity or price is invalid, because those counts do not require these numbers.


In [7]:
orders["quantity_value"] = pd.array(
    raw_df["quantity"].map(validate_quantity), dtype="Int64"
)
# Object dtype retains valid Decimal values and uses None for invalid prices.
orders["unit_price_value"] = pd.Series(
    [validate_unit_price(value) for value in raw_df["unit_price"]],
    index=orders.index, dtype="object",
)
orders["invalid_quantity"] = orders["quantity_value"].isna()
orders["invalid_unit_price"] = orders["unit_price_value"].isna()
orders["numeric_problem"] = (
    orders["invalid_quantity"] | orders["invalid_unit_price"]
)

numeric_summary = pd.Series({
    "Invalid quantity": int(orders["invalid_quantity"].sum()),
    "Invalid unit price": int(orders["invalid_unit_price"].sum()),
    "Orders with either numeric problem": int(orders["numeric_problem"].sum()),
    "Orders with both numeric problems": int(
        (orders["invalid_quantity"] & orders["invalid_unit_price"]).sum()
    ),
}, name="orders")
display(numeric_summary.to_frame())
display(orders.loc[orders["numeric_problem"], [
    "order_id", "quantity", "unit_price", "invalid_quantity", "invalid_unit_price"
]])


,orders
Invalid quantity,7
Invalid unit price,12
Orders with either numeric problem,19
Orders with both numeric problems,0


,order_id,quantity,unit_price,invalid_quantity,invalid_unit_price
23,1500,3,,False,True
164,630,-1,22,True,False
199,625,2,,False,True
216,1000,1,,False,True
239,1250,1,,False,True
293,500,1,,False,True
503,375,3,,False,True
571,250,4,,False,True
583,125,1,,False,True
672,840,-1,23.32,True,False


## Sprint 4 — Business rules, dates, and exploration

### 8. Test the delivery hypothesis

**Hypothesis:** every order without a delivery date is cancelled.

This is a one-way statement. It does not claim that every cancelled order must
have a blank delivery date. We inspect the recorded status using a temporary
trimmed, lowercase comparison, without changing the status column. A single
noncancelled order without a delivery date disproves the hypothesis.


In [8]:
def investigate_delivery(frame):
    """Return the cancellation mask and the two groups without delivery dates."""
    missing_delivery = frame["delivery_date"].map(is_blank)
    cancelled = frame["order_status"].str.strip().str.lower().eq("cancelado")
    without_delivery_cancelled = frame.loc[missing_delivery & cancelled].copy()
    without_delivery_other = frame.loc[missing_delivery & ~cancelled].copy()
    return cancelled, without_delivery_cancelled, without_delivery_other


cancelled, no_delivery_cancelled, no_delivery_other = investigate_delivery(orders)

print(f"Total cancelled orders: {cancelled.sum()}")
print(f"No delivery date and cancelled: {len(no_delivery_cancelled)}")
print(f"No delivery date and another status: {len(no_delivery_other)}")

if len(no_delivery_other) > 0:
    print("Hypothesis: FALSE. Counterexamples exist in this dataset.")
    display(no_delivery_other[["order_id", "order_status", "delivery_date"]].head(3))
else:
    print("Hypothesis: TRUE in this dataset. No counterexample was found.")


Total cancelled orders: 109
No delivery date and cancelled: 109
No delivery date and another status: 34
Hypothesis: FALSE. Counterexamples exist in this dataset.


,order_id,order_status,delivery_date
49,1170,Concluído,
117,990,Concluído,
152,740,Concluído,


### 9. Parse sale timestamps without stopping on errors

`datetime.strptime` interprets the required `YYYY-MM-DD HH:MM:SS` format.
`try/except` lets later rows continue after an invalid date. A missing or invalid
date returns `None`; it is never replaced with today's date.

The source timestamps represent local São Paulo time with no explicit time
zone. We preserve that interpretation and do not convert time zones. The
original `sale_date` text remains available for inspection.


In [9]:
def parse_sale_date(value):
    """Parse the specified local timestamp or return None for a date problem."""
    if is_blank(value):
        return None
    try:
        return datetime.strptime(str(value).strip(), DATE_FORMAT)
    except ValueError:
        return None


# A list and an explicit loop make the row-by-row error handling easy to follow.
parsed_dates = []
for value in raw_df["sale_date"]:
    parsed_dates.append(parse_sale_date(value))

orders["sale_datetime"] = pd.Series(parsed_dates, index=orders.index, dtype="object")
orders["invalid_sale_date"] = orders["sale_datetime"].isna()

for part in ["year", "month", "day", "hour"]:
    orders[f"sale_{part}"] = pd.array(
        [getattr(value, part) if value is not None else None for value in parsed_dates],
        dtype="Int64",
    )

orders["sale_date_br"] = [
    value.strftime("%d/%m/%Y") if value is not None else None
    for value in parsed_dates
]
orders["sale_year_month"] = [
    value.strftime("%Y-%m") if value is not None else None
    for value in parsed_dates
]

print(f"Invalid or blank sale dates: {orders['invalid_sale_date'].sum()}")
display(orders[[
    "order_id", "sale_date", "sale_datetime", "sale_year", "sale_month",
    "sale_day", "sale_hour", "sale_date_br"
]].head(3))
display(orders.loc[orders["invalid_sale_date"], ["order_id", "sale_date"]])


Invalid or blank sale dates: 4


,order_id,sale_date,sale_datetime,sale_year,sale_month,sale_day,sale_hour,sale_date_br
0,844,2025-11-21 18:02:35,2025-11-21 18:02:35,2025,11,21,18,21/11/2025
1,378,2024-10-29 18:41:26,2024-10-29 18:41:26,2024,10,29,18,29/10/2024
2,953,2025-09-05 21:16:09,2025-09-05 21:16:09,2025,9,5,21,05/09/2025


,order_id,sale_date
152,740,data_invalida
255,370,data_invalida
714,1480,data_invalida
884,1110,data_invalida


### 10. Count orders with the correct inclusion rules

Channel counts require a recognized channel. Year-month counts require a valid
sale date. Both include **all statuses**, including cancelled and returned
orders. Each count reports its own exclusions. Numeric problems do not exclude
an order from these counts. Grouping by year-month keeps 2024 and 2025 separate.


In [10]:
def count_orders(frame, column, eligible):
    """Count eligible rows by a field; every row represents one order."""
    return frame.loc[eligible, column].value_counts().rename("orders")


recognized_channel = ~orders["unrecognized_channel"]
valid_sale_date = ~orders["invalid_sale_date"]

channel_counts = count_orders(orders, "sales_channel", recognized_channel)
period_counts = count_orders(orders, "sale_year_month", valid_sale_date).sort_index()

print("Orders by channel — all statuses included")
print(f"Included: {recognized_channel.sum()}; excluded: {(~recognized_channel).sum()}")
display(channel_counts.to_frame())

print("Orders by year-month — all statuses included")
print(f"Included: {valid_sale_date.sum()}; excluded: {(~valid_sale_date).sum()}")
display(period_counts.to_frame())


Orders by channel — all statuses included
Included: 1500; excluded: 0


,orders
sales_channel,
Web,666
Loja física,551
Mercado,283


Orders by year-month — all statuses included
Included: 1496; excluded: 4


,orders
sale_year_month,
2024-08,77
2024-09,104
2024-10,393
2024-11,174
2025-08,62
2025-09,94
2025-10,410
2025-11,182


### 11. Produce two evidence-based findings

Every finding needs numbers, interpretation, and a limitation. Percentages use
the relevant included population. The October comparison concerns recorded
orders, not completed sales or revenue. A seasonal association does not prove
that Halloween caused the difference.


In [11]:
leading_channel = channel_counts.idxmax()
leading_channel_orders = int(channel_counts.max())
channel_share = 100 * leading_channel_orders / int(channel_counts.sum())

october_2024 = int(period_counts.get("2024-10", 0))
october_2025 = int(period_counts.get("2025-10", 0))
october_total = october_2024 + october_2025
october_share = 100 * october_total / int(period_counts.sum())

print("Finding 1 — Channel")
print(
    f"{leading_channel} leads with {leading_channel_orders} orders "
    f"({channel_share:.2f}% of orders with a recognized channel)."
)
print("Interpretation: the website is the most-used ordering channel in this sample.")
print(
    "Limitation: all statuses are included; this is not a measure of revenue, "
    "completed sales, or the real costume market."
)

print("\nFinding 2 — Period")
print(
    f"October has {october_2024} orders in 2024 and {october_2025} in 2025: "
    f"{october_total} combined ({october_share:.2f}% of valid-dated orders)."
)
print("Interpretation: orders concentrate in October in both observed years.")
print(
    "Limitation: four invalid dates are excluded; only August-November of two "
    "years is observed, so this does not establish full-year seasonality or causation."
)


Finding 1 — Channel
Web leads with 666 orders (44.40% of orders with a recognized channel).
Interpretation: the website is the most-used ordering channel in this sample.
Limitation: all statuses are included; this is not a measure of revenue, completed sales, or the real costume market.

Finding 2 — Period
October has 393 orders in 2024 and 410 in 2025: 803 combined (53.68% of valid-dated orders).
Interpretation: orders concentrate in October in both observed years.
Limitation: four invalid dates are excluded; only August-November of two years is observed, so this does not establish full-year seasonality or causation.


## Sprint 5 — Final report and documentation

### 12. Display the manual status report

This dictionary assembles the required counts explicitly. A numeric-field
problem and an invalid date may affect the same row. The final combined total
uses a Boolean union again, so it does not add overlapping occurrences.

Category and channel normalization are reported separately from unresolved
numeric and date problems. No record is discarded from the working DataFrame.


In [12]:
orders["validation_problem"] = (
    orders["numeric_problem"] | orders["invalid_sale_date"]
    | orders["unrecognized_channel"]
)

summary = {
    "Rows read": len(raw_df),
    "Blank categories filled": int(originally_blank_category.sum()),
    "Nonblank categories standardized": int(changed_nonblank_category.sum()),
    "Recognized channel values standardized": int(channel_changes.sum()),
    "Invalid quantities": int(orders["invalid_quantity"].sum()),
    "Invalid unit prices": int(orders["invalid_unit_price"].sum()),
    "Orders with a numeric problem (union)": int(orders["numeric_problem"].sum()),
    "Invalid or blank sale dates": int(orders["invalid_sale_date"].sum()),
    "Orders with numeric/date/channel problems (union)": int(orders["validation_problem"].sum()),
    "Total cancelled orders": int(cancelled.sum()),
    "No delivery date and cancelled": len(no_delivery_cancelled),
    "No delivery date and another status": len(no_delivery_other),
    "Excluded from channel counts": int((~recognized_channel).sum()),
    "Excluded from year-month counts": int((~valid_sale_date).sum()),
}
display(pd.Series(summary, name="count").to_frame())


,count
Rows read,1500
Blank categories filled,18
Nonblank categories standardized,1482
Recognized channel values standardized,45
Invalid quantities,7
Invalid unit prices,12
Orders with a numeric problem (union),19
Invalid or blank sale dates,4
Orders with numeric/date/channel problems (union),23
Total cancelled orders,109


### 13. Reflect on data quality and Machine Learning

Missing prices and invalid quantities could distort numeric features. Inconsistent
category and channel text could split one real category into several labels.
Invalid dates could distort time-based comparisons. Keeping original values and
marking problems makes our decisions explainable and avoids invented evidence.

Cleaning alone does **not** prevent overfitting or underfitting. Representative
data, an appropriate model, suitable features, and evaluation on unseen data
also matter. A future model using time-dependent orders would need an appropriate
time-based evaluation and safeguards against data leakage. This project does not
train a model; its task is data preparation and exploratory analysis.

The dataset is fictional and deliberately contains errors. The findings describe
these registered orders, not the real market. The delivery counterexamples show
that missing delivery information cannot safely be used to infer cancellation.


### 14. Check the final written report

`report.md` contains the required summary and findings. Read the exact UTF-8
text, including Markdown, spaces, and line breaks, to check the 1,500–3,000
character requirement. The AI assistant used this deterministic count to verify
the final text; the code makes that check reproducible.

This cell reads the report; it never rewrites it. If you edit the report, rerun
the check. The README contains the execution guide, results, source reference,
evaluation checklist, branch explanation, and a 5–7 minute presentation outline.


In [13]:
report_text = Path("report.md").read_text(encoding="utf-8")
report_character_count = len(report_text)
print(f"report.md characters, including spaces and line breaks: {report_character_count:,}")
if 1500 <= report_character_count <= 3000:
    print("Report length: meets the assignment requirement.")
else:
    print("Report length: revise the report to contain 1,500–3,000 characters.")
print("\n" + report_text)


report.md characters, including spaces and line breaks: 2,548
Report length: meets the assignment requirement.

# Halloween Orders: Findings and Insights

The analysis covers 1,500 distinct orders from the fictional retailer Festa em Cena, recorded from August to November in 2024 and 2025. Each row represents one order for one product, possibly with several units. Counts describe registered orders across all statuses, not revenue, units sold, or completed sales.

The notebook uses Pandas to read halloween_sales.csv as UTF-8 text, with all 14 column names already in English. The input CSV is preserved without overwriting its values. Category cleaning fills 18 originally blank values with the required label "sem categoria" and changes 1,482 nonblank values through trimming, lowercasing, and punctuation removal. Sales channels are standardized to Mercado, Loja física, and Web; 45 channel values change and all channels are recognized.

Numeric validation flags 7 invalid quantities and 12 m